In [ ]:
%load_ext autoreload
%autoreload 2

import anndata as ad
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load the AnnData object
adata = ad.read_h5ad("./data/fucci/rpe1_kinetics_processed.h5ad")
adata

In [ ]:
s_genes = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM7", "MCM4", "RRM1", "UNG", "GINS2", "MCM6",
    "CDCA7", "DTL", "PRIM1", "UHRF1", "CENPU", "HELLS", "RFC2", "POLR1B", "NASP",
    "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7", "POLD3", "MSH2", "ATAD2",
    "RAD51", "RRM2", "CDC45", "CDC6", "EXO1", "TIPIN", "DSCC1", "BLM", "CASP8AP2",
    "USP1", "CLSPN", "POLA1", "CHAF1B", "MRPL36", "E2F8"
]

g2m_genes = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80", "CKS2", "NUF2",
    "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "PIMREG", "SMC4", "CCNB2", "CKAP2L",
    "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E", "TUBB4B", "GTSE1", "KIF20B", "HJURP",
    "CDCA3", "JPT1", "CDC20", "TTK", "CDC25C", "KIF2C", "RANGAP1", "NCAPD2", "DLGAP5",
    "CDCA2", "CDCA8", "ECT2", "KIF23", "HMMR", "AURKA", "PSRC1", "ANLN", "LBR", "CKAP5",
    "CENPE", "CTCF", "NEK2", "G2E3", "GAS2L3", "CBX5", "CENPA"
]

# --- Step 1: Filter all genes globally ---

# Extract all genes (not just cell cycle subset)
X_all = adata.layers["X_total"].toarray()
V_all = adata.layers["velocity_T"].toarray()

# Expression filter
avg_expr = X_all.mean(axis=0)
expr_threshold = np.quantile(avg_expr, 0.2)
expr_mask = avg_expr > expr_threshold

# Velocity filter
nonzero_velocity_mask = (V_all != 0).any(axis=0)

# Global mask
global_mask = expr_mask & nonzero_velocity_mask

X_filtered = X_all[:, global_mask]
V_filtered = V_all[:, global_mask]
genes_filtered = adata.var_names[global_mask]

phase_numeric = [float(i) for i in list(adata.obs["Cell_cycle_relativePos"])]
phase = list(adata.obs["cell_cycle_phase"])

# --- Step 2: Transform ---
X_log1p = np.log1p(X_filtered)
X_log1p = X_log1p - X_log1p.mean(axis=0, keepdims=True)
V_std = V_filtered / V_filtered.std(axis=0, ddof=0)

# --- Step 3: Subset to cell cycle genes ---
cell_cycle_genes = s_genes + g2m_genes
genes_present = [g for g in cell_cycle_genes if g in genes_filtered]

# Indices of those within the filtered set
gene_indices = [np.where(genes_filtered == g)[0][0] for g in genes_present]

X_cc = X_log1p[:, gene_indices]
V_cc = V_std[:, gene_indices]
genes_cc = np.array(genes_present)

X_cc.shape, V_cc.shape, len(genes_cc)

In [ ]:
from scripts.VectorFieldEmbedder import *
from scripts.plotting import *
from scripts.TPS import *

emb = VectorFieldEmbedder(X_log1p, V_std, dist_method="phase",
                          embed_kwargs={"n_neighbors":30,
                                        "min_dist":0.3},
                          dof=30,method="umap")
emb.initialize_embedding(42)

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)

In [ ]:
from scripts.VectorFieldEmbedder import *
from scripts.plotting import *
from scripts.TPS import *

emb = VectorFieldEmbedder(X_cc, V_cc, dist_method="phase",
                          embed_kwargs={"n_neighbors":30,
                                         "min_dist":0.6},
                          dof=30,method="umap")
emb.initialize_embedding(1)

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0,
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    show_colorbar=False
)

In [ ]:
emb.optimize()

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.7,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    # show_colorbar=True
)

In [ ]:
import os
import pickle

# --------------------------
# Save embedding object
# --------------------------
outdir = "./figures/cell_cycle"
os.makedirs(outdir, exist_ok=True)

with open(f"{outdir}/flowmap_embedding.pkl", "wb") as f:
    pickle.dump(emb, f)

print("Saved embedding object to ./figures/cell_cycle/flowmap_embedding.pkl")


In [ ]:
import matplotlib.patheffects as pe

def plot_velocity_streamplot(
    X_2d, tps_vf=None, V=None, scatter_color="grey",
    grid_size=50, grid_density=1.0, stream_density=1.0, title=None,
    scatter_size=10, scatter_alpha=0.5, arrowsize=1.5,
    ax=None, figsize=(8, 6), aspect="equal", cmap="tab10",
    vmin=None, vmax=None, show_axes=True, show_colorbar=False,
    streamline_thickness=4.0, show_labels=True, use_cmap=True, cell_type=None,
):

    # --- fit model if not provided ---
    if tps_vf is None:
        if V is None:
            raise ValueError("Either tps_vf or V must be provided.")
        from .TPS import ThinPlateSpline

        # --- subsample if too many points ---
        n_points = X_2d.shape[0]
        max_points = 4000
        if n_points > max_points:
            idx = np.random.choice(n_points, max_points, replace=False)
            X_fit = X_2d[idx]
            V_fit = V[idx]
            print(f"[TPS] Subsampling {max_points}/{n_points} points for fitting …")
        else:
            X_fit = X_2d
            V_fit = V
            print(f"[TPS] Using all {n_points} points for fitting …")

        # --- fit thin-plate spline on 2D subset ---
        tps_vf = ThinPlateSpline(X_fit, n_control_points=100)
        tps_vf.fit(V_fit, dof=15)

    # --- compute filtered grid ---
    Xg, keep, Vg, (xx, yy) = compute_velocity_on_grid(
        X_2d, tps_vf=tps_vf,
        grid_size=grid_size, grid_density=grid_density,
        min_mass=0.01, return_mesh=True
    )
    
    # --- reconstruct full grid ---
    ny, nx = yy.shape[0], xx.shape[1]
    grid_x, grid_y = xx[0, :], yy[:, 0]
    Vx = np.full((ny, nx), np.nan)
    Vy = np.full((ny, nx), np.nan)

    dx = (grid_x[-1] - grid_x[0]) / (nx - 1)
    dy = (grid_y[-1] - grid_y[0]) / (ny - 1)
    j_idx = np.clip(np.rint((Xg[:, 0] - grid_x[0]) / dx).astype(int), 0, nx - 1)
    i_idx = np.clip(np.rint((Xg[:, 1] - grid_y[0]) / dy).astype(int), 0, ny - 1)
    Vx[i_idx, j_idx] = Vg[:, 0]
    Vy[i_idx, j_idx] = Vg[:, 1]

    U = np.ma.masked_invalid(Vx)
    V = np.ma.masked_invalid(Vy)

    # --- plotting ---
    created_fig = False
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
        created_fig = True

    # ---------- robust scatter coloring ----------
    scatter_color = np.array(scatter_color)
    if scatter_color.dtype.kind in {"U", "S", "O"}:  # categorical labels
        unique_vals = np.unique(scatter_color)
        cmap_obj = plt.get_cmap(cmap, len(unique_vals))
        color_map = {val: mcolors.to_hex(cmap_obj(i)) for i, val in enumerate(unique_vals)}
        mapped_colors = np.array([color_map[val] for val in scatter_color])
        ax.scatter(
            X_2d[:, 0], X_2d[:, 1],
            s=scatter_size, alpha=scatter_alpha,
            edgecolors="none", color=mapped_colors
        )
    else:  # numeric or pre-colored array
        if np.issubdtype(scatter_color.dtype, np.number):
            sc = ax.scatter(
                X_2d[:, 0], X_2d[:, 1],
                s=scatter_size, alpha=scatter_alpha,
                c=scatter_color, cmap=cmap, vmin=vmin, vmax=vmax,
                edgecolors="none"
            )
            if show_colorbar:
                plt.colorbar(sc, ax=ax)
        else:
            ax.scatter(
                X_2d[:, 0], X_2d[:, 1],
                s=scatter_size, alpha=scatter_alpha,
                edgecolors="none", color=scatter_color
            )
    # ---------------------------------------------

    # --- streamlines ---
    speed = np.sqrt(Vx**2 + Vy**2)
    smax = np.nanmax(speed) if np.isfinite(speed).any() else 0.0
    linewidth = streamline_thickness * (speed / smax) if smax > 0 else 1.0

    ax.streamplot(
        grid_x, grid_y, U, V,
        linewidth=linewidth,
        density=stream_density,
        color="k",
        arrowsize=arrowsize,
        arrowstyle="-|>",
        maxlength=4,
        integration_direction="both"
    )

    # ----- cell type labels -----
    if cell_type is not None:
        cell_type = np.asarray(cell_type)
        for ct in np.unique(cell_type):
            idx = cell_type == ct
            if idx.sum() == 0:
                continue

            # center of the cluster
            x_center = np.median(X_2d[idx, 0])
            y_center = np.median(X_2d[idx, 1])

            txt = plt.text(
                x_center,
                y_center,
                str(ct),
                ha="center",
                va="center",
                fontsize=18,
                color="black",
                weight="bold",
                zorder=10,
            )

            # white boundary / outline
            txt.set_path_effects([
                pe.Stroke(linewidth=5.5, foreground="white"),
                pe.Normal(),
            ])

    
    ax.set_aspect(aspect)
    if not show_axes:
        ax.set_xticks([]); ax.set_yticks([]); ax.set_frame_on(False)
    else:
        ax.grid(True, linestyle='--', alpha=0.3)
    if title:
        ax.set_title(title)

    if created_fig:
        plt.tight_layout()
        plt.show()


plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.9,
    scatter_size=100,
    scatter_alpha=0.1,
    streamline_thickness=8.0,
    figsize=(4, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    cmap="viridis",
    cell_type=phase,
)

In [ ]:
from scripts.plotting import *
from scripts.FieldReconstructionEvaluator import *

emb.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50, X=X_log1p, V=V_std)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()

In [ ]:
emb.gene_names = genes_cc
plot_gene_r2_scatter(res, 
                     genes_filtered,
                     thr_expr=0.5,
                     thr_vel=0.3)

In [ ]:
emb.fit_gene_level_splines(dof_gene=15, dof_vf_gene=15, X=emb.X, V=emb.V)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()
print(res['expr_var_explained'], res['vel_var_explained'])

emb.fit_gene_level_splines(dof_gene=15, dof_vf_gene=15, X=X_log1p, V=V_std)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()
print(res['expr_var_explained'], res['vel_var_explained'])

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# DOF sweep
# ------------------------------------------------------------
dofs = [10, 20, 30, 50, 75, 100]

expr_var = []
vel_var = []

for dof in dofs:
    print(f"[DOF={dof}] fitting gene-level splines ...")

    emb.fit_gene_level_splines(
        dof_gene=dof,
        dof_vf_gene=dof,
        X=emb.X,
        V=emb.V
    )

    evaluator = FieldReconstructionEvaluator(emb)
    res = evaluator.evaluate_gene_fit()

    expr_var.append(res["expr_var_explained"])
    vel_var.append(res["vel_var_explained"])

    print(
        f"  expr R2 = {res['expr_var_explained']:.3f}, "
        f"vel R2 = {res['vel_var_explained']:.3f}"
    )

expr_var = np.array(expr_var)
vel_var  = np.array(vel_var)

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
plt.figure(figsize=(7, 6))

plt.plot(
    dofs, expr_var,
    marker="o", lw=2.5, ms=8,
    label="Expression"
)

plt.plot(
    dofs, vel_var,
    marker="s", lw=2.5, ms=8,
    label="Velocity"
)

plt.xlabel("Spline degrees of freedom")
plt.ylabel("Variance explained ($R^2$)")

plt.grid(alpha=0.3)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import math


emb.fit_gene_level_splines(dof_gene=50, dof_vf_gene=50, X=X_log1p, V=V_std)
evaluator = FieldReconstructionEvaluator(emb)
res = evaluator.evaluate_gene_fit()

# --- make fonts larger globally ---
plt.rcParams.update({
    "font.size": 22,
    "axes.labelsize": 24,
    "axes.titlesize": 26,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "legend.fontsize": 18
})

# ------------------------------------------------------------------
# 1) INCLUDE ALL GENES
# ------------------------------------------------------------------
x_all = np.array(res["expr_corr_gene"])
y_all = np.array(res["vel_corr_gene"])
gene_names_all = adata.var_names[global_mask]

mask = np.isfinite(x_all) & np.isfinite(y_all)

x = x_all[mask]
y = y_all[mask]
gene_names = gene_names_all[mask]

# ------------------------------------------------------------------
# 2) DEFINE CURATED CELL-CYCLE GENES
# ------------------------------------------------------------------
genes_cc = np.array(genes_cc)                 # list of 33 gene names
genes_cc_upper = set(g.upper() for g in genes_cc)

categories = np.array([
    "Cell-cycle genes (n=33)" if g.upper() in genes_cc_upper else "Other genes"
    for g in gene_names
])

ordered_categories = [
    "Cell-cycle genes (n=33)",
    "Other genes"
]

categories = pd.Categorical(
    categories,
    categories=ordered_categories,
    ordered=True
)

# Masks
is_cc = categories == "Cell-cycle genes (n=33)"
is_other = categories == "Other genes"

# ------------------------------------------------------------------
# 3) PLOT
# ------------------------------------------------------------------
sns.set_style("whitegrid")
g = sns.JointGrid(x=x, y=y, height=7)

# ----------------------------
# Background: Other genes
# ----------------------------
sns.scatterplot(
    x=x[is_other],
    y=y[is_other],
    color="lightgrey",
    s=60,
    alpha=0.4,
    edgecolor="none",
    ax=g.ax_joint,
    legend=False
)

# ----------------------------
# Foreground: Cell-cycle genes
# ----------------------------
sns.scatterplot(
    x=x[is_cc],
    y=y[is_cc],
    color="#d62728",          # clean red (matches many cell-cycle figs)
    s=180,
    edgecolor="white",
    linewidth=0.6,
    alpha=0.9,
    ax=g.ax_joint,
    label="Cell-cycle \ngenes (n=33)"
)

# Regression line (all genes)
sns.regplot(
    x=x,
    y=y,
    scatter=False,
    color="crimson",
    line_kws={"lw": 3, "alpha": 0.8},
    ax=g.ax_joint
)

# Marginals
sns.histplot(
    x=x,
    bins=30,
    element="step",
    color="grey",
    alpha=0.5,
    ax=g.ax_marg_x
)
sns.histplot(
    y=y,
    bins=30,
    element="step",
    color="grey",
    alpha=0.5,
    ax=g.ax_marg_y
)

# Reference lines
g.ax_joint.axhline(0, color="grey", ls="--", lw=1.5, alpha=0.6)
g.ax_joint.axvline(0, color="grey", ls="--", lw=1.5, alpha=0.6)

g.set_axis_labels(
    "Expression: Pearson $r$",
    "Velocity: Pearson $r$"
)

# ------------------------------------------------------------------
# 4) LEGEND (simple, clean)
# ------------------------------------------------------------------
g.ax_joint.legend(
    loc="upper left",
    # bbox_to_anchor=(0.5, -0.16),
    frameon=False,
    ncol=1,
    handletextpad=0.4,
    borderaxespad=0.0
)

plt.tight_layout()
plt.show()

In [ ]:
plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.0,
    scatter_size=20,
    scatter_alpha=0.8,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    show_colorbar=True,
    cmap="viridis"
)

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=phase_numeric,
    grid_density=1.0, 
    stream_density=0.0,
    scatter_size=20,
    scatter_alpha=0.8,
    figsize=(5, 4),
    aspect=1,
    grid_size=50,
    show_axes=False,
    show_colorbar=True,
    cmap="viridis"
)

In [ ]:
marker_genes = ["MKI67", "PCNA", "CCNB1", "CDK1",
                "TOP2A", "CDT1", "E2F1", "AURKA"]  # example markers

n_cols = 4
n_rows = int(np.ceil(len(marker_genes) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))

for idx, gene in enumerate(marker_genes):
    row, col = divmod(idx, n_cols)
    ax = axes[row, col] if n_rows > 1 else axes[col]
    
    if gene in adata.var_names:
        scatter_color = adata[:, gene].X.toarray().flatten()
    else:
        scatter_color = np.zeros(adata.n_obs)  # fallback

    plot_velocity_streamplot(
        X_2d=emb.X_emb,
        tps_vf=emb.tps_vf,
        scatter_color=scatter_color,
        grid_density=1.0, 
        stream_density=0.0,
        scatter_size=35,        # bigger points
        scatter_alpha=0.85,
        figsize=(5, 4),
        aspect=1,
        grid_size=50,
        show_axes=False,
        show_colorbar=False,    # no colorbar
        ax=ax,
        cmap="viridis",
        title=None              # we’ll set title manually
    )
    
    ax.set_title(gene, fontsize=20, pad=10)

# remove empty axes if not multiple of 4
for j in range(len(marker_genes), n_rows * n_cols):
    fig.delaxes(axes.flatten()[j])

plt.tight_layout()
plt.show()


In [ ]:
from scripts.VectorFieldGeometry import *

fps = find_fixed_points_grid(
    emb.tps_vf,
    emb.X_emb,
    tol_vec_percent=0.01,
    tol_merge_percent=0.05,
    grid_size=120
)

Xg, keep, Vg = compute_velocity_on_grid(
    emb.X_emb,
    tps_vf=emb.tps_vf,
    grid_size=120,
    grid_density=1.0,
    min_mass=0.01,
    return_mesh=False
)

# 3) Loop through candidate fixed points
labels = []
for i, fp in enumerate(fps):
    try:
        J = jacobian_from_data(
            emb.tps_vf,
            fp,
            Xg,
            Vg,
            radius_percent=0.1
        )
        label = classify_fixed_point(J)
    except ValueError:
        label = "Insufficient data for fit"

    labels.append(label)
    print(f"FP{i}: {np.round(fp, 3)}  →  {label}")

In [ ]:
import numpy as np
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# embedding and ground truth
X = emb.X_emb
gt = phase_numeric

# chosen center
center = fps[1,]

# --- plot embedding with center ---
plt.figure(figsize=(6, 6))
sc = plt.scatter(
    X[:,0], X[:,1],
    s=15, alpha=0.7, c=gt, cmap="viridis"
)
plt.scatter(
    center[0], center[1],
    c="red", s=200, marker="x", label=label
)
# plt.colorbar(sc, label="Ground truth position")
plt.legend()
plt.axis("equal")
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1) Prepare angles (ground truth + embedding)
# ============================================================

X = emb.X_emb
gt = np.array(phase_numeric)            # assumed in [0,1]
center = fps[1, :]

# embedding angles
X_shifted = X - center
theta_emb = np.arctan2(X_shifted[:, 1], X_shifted[:, 0])
theta_emb = (theta_emb + 2 * np.pi) % (2 * np.pi)

# ground truth angles
theta_gt = (gt % 1.0) * 2 * np.pi


# ============================================================
# 2) Circular–circular correlation (Jammalamadaka–Sengupta)
# ============================================================

def circular_correlation(alpha, beta):
    """
    Circular–circular correlation coefficient.
    alpha, beta in radians.
    """
    alpha_bar = np.angle(np.mean(np.exp(1j * alpha)))
    beta_bar  = np.angle(np.mean(np.exp(1j * beta)))

    num = np.sum(
        np.sin(alpha - alpha_bar) * np.sin(beta - beta_bar)
    )
    den = np.sqrt(
        np.sum(np.sin(alpha - alpha_bar) ** 2) *
        np.sum(np.sin(beta - beta_bar) ** 2)
    )
    return num / den


rho = circular_correlation(theta_emb, theta_gt)
print(f"Circular–circular correlation ρ = {rho:.3f}")


# ============================================================
# 3) Polar scatter plot (vanilla cosmetics, same labeling)
# ============================================================

fig = plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

sc = ax.scatter(
    theta_emb,
    theta_gt,
    c=gt,
    cmap="viridis",
    s=12,          # smaller markers
    alpha=0.7
)

# ---- angular convention ----
# 180 degrees (pi) -> 0
# clockwise increase
ax.set_theta_zero_location("W")   # 180° as 0
ax.set_theta_direction(-1)        # clockwise

# label angular ticks as cycle position [0, 1]
ticks = np.linspace(0, 2 * np.pi, 5)
ax.set_thetagrids(
    np.degrees(ticks),
    labels=["0;1", "0.25", "0.5", "0.75", ""]
)


# ---- radial axis cleanup ----
ax.set_yticklabels([])
ax.set_ylabel("")

# ---- title (simple) ----
ax.set_title(
    f"|circular corr| = {abs(rho):.2f}",
    pad=10
)

# ---- colorbar (compact) ----
cbar = plt.colorbar(sc, pad=0.2, shrink=0.5)
cbar.set_label("Ground truth cycle position")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# JS circular–circular correlation
# ============================================================

def circular_correlation_js(alpha, beta):
    """
    Jammalamadaka–Sengupta circular–circular correlation coefficient.
    alpha, beta: angles in radians.
    """
    alpha_bar = np.angle(np.mean(np.exp(1j * alpha)))
    beta_bar  = np.angle(np.mean(np.exp(1j * beta)))

    num = np.sum(
        np.sin(alpha - alpha_bar) * np.sin(beta - beta_bar)
    )
    den = np.sqrt(
        np.sum(np.sin(alpha - alpha_bar) ** 2) *
        np.sum(np.sin(beta - beta_bar) ** 2)
    )
    return num / den


# ============================================================
# Prepare angles
# ============================================================

X = emb.X_emb
gt = np.array(phase_numeric)          # in [0,1]
center = fps[1, :]

# embedding angle
X_shifted = X - center
theta_emb = np.arctan2(X_shifted[:, 1], X_shifted[:, 0])
theta_emb = (theta_emb + 2*np.pi) % (2*np.pi)

# ground-truth angle
theta_gt = (gt % 1.0) * 2*np.pi


# ============================================================
# JS circular correlation
# ============================================================

circ_corr = circular_correlation_js(theta_emb, theta_gt)
print(f"JS circular correlation = {circ_corr:.3f}")


# ============================================================
# Circular difference after global alignment
# ============================================================

# raw circular difference
delta_raw = np.angle(np.exp(1j * (theta_emb - theta_gt)))

# global phase offset (mean rotation)
phase_offset = np.angle(np.mean(np.exp(1j * delta_raw)))

# aligned difference
delta_aligned = np.angle(np.exp(1j * (delta_raw - phase_offset)))


# ============================================================
# Plot Δθ histogram (vanilla, supplement style)
# ============================================================

plt.figure(figsize=(8, 6))

plt.hist(
    delta_aligned,
    bins=40,
    density=True,
    color="steelblue",
    alpha=0.85
)

plt.axvline(0, color="black", lw=2, ls="--")

plt.xlabel(r"Aligned circular difference $\Delta\theta$ (rad)")
plt.ylabel("Density")
plt.title(f"|JS circular corr| = {abs(circ_corr):.2f}")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.cluster.hierarchy import linkage, leaves_list

# ============================================================
# 0. Utilities
# ============================================================

def zscore(X):
    """Per-gene z-score (columns). For visualization safety."""
    return (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

def cluster_genes(X):
    """Cluster genes (columns) using Ward linkage."""
    link = linkage(X.T, method="ward")
    return X[:, leaves_list(link)]

# ============================================================
# 1. Cell order & gene masks
# ============================================================

order = np.argsort(phase_numeric)

expr_corr = np.asarray(res["expr_corr_gene"])
vel_corr  = np.asarray(res["vel_corr_gene"])

good_mask = (expr_corr > 0.3) & (vel_corr > 0.3)
bad_mask  = (expr_corr < 0.3) & (vel_corr < 0.3)

# ============================================================
# 2. Extract expression (already normalized upstream,
#    but re-normalized here for visualization robustness)
# ============================================================

X_good = emb.X_gene[order][:, good_mask]
X_bad  = emb.X_gene[order][:, bad_mask]

X_good = zscore(X_good)
X_bad  = zscore(X_bad)

# ============================================================
# 3. Circular smoothing along phase order
# ============================================================

X_good = gaussian_filter1d(X_good, sigma=10, axis=0, mode="wrap")
X_bad  = gaussian_filter1d(X_bad,  sigma=10, axis=0, mode="wrap")

# ============================================================
# 4. Cluster genes independently
# ============================================================

X_good = cluster_genes(X_good)
X_bad  = cluster_genes(X_bad)

# ============================================================
# 5. Circular phase boundary inference
# ============================================================

phases = np.asarray(phase)[order]
n_cells = len(phases)

theta = np.linspace(0, 2*np.pi, n_cells, endpoint=False)

phase_progression = ["S", "G2-M", "M", "M-G1", "G1-S"]

def circular_mean(angles):
    return np.arctan2(
        np.mean(np.sin(angles)),
        np.mean(np.cos(angles))
    ) % (2*np.pi)

# Mean angle per phase
phase_angle = {}
for p in phase_progression:
    mask = phases == p
    phase_angle[p] = circular_mean(theta[mask])

# Order phases by angular position
ordered_phases = sorted(phase_progression, key=lambda p: phase_angle[p])

# Anchor + unwrap
anchor = ordered_phases[0]
anchor_angle = phase_angle[anchor]

def unwrap(a):
    return (a - anchor_angle) % (2*np.pi)

phase_angle_u = {p: unwrap(phase_angle[p]) for p in ordered_phases}

# Progressive right boundaries
boundary_angles = []
current = []
for p in ordered_phases:
    current.append(p)
    right = max(phase_angle_u[q] for q in current)
    boundary_angles.append((right + anchor_angle) % (2*np.pi))

# ============================================================
# 6. Polar heatmap visualization
# ============================================================

fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="polar")

theta_edges = np.linspace(0, 2*np.pi, n_cells + 1)

# ----------------------------
# Inner track: bad genes
# ----------------------------
r_bad_inner, r_bad_outer = 0.58, 0.75
r_bad_edges = np.linspace(r_bad_inner, r_bad_outer, X_bad.shape[1] + 1)
T_bad, R_bad = np.meshgrid(theta_edges, r_bad_edges)

ax.pcolormesh(
    T_bad, R_bad, X_bad.T,
    cmap="RdBu_r",
    vmin=-1.5, vmax=1.5,
    shading="flat"
)

# ----------------------------
# Outer track: good genes
# ----------------------------
r_good_inner, r_good_outer = 0.83, 1.0
r_good_edges = np.linspace(r_good_inner, r_good_outer, X_good.shape[1] + 1)
T_good, R_good = np.meshgrid(theta_edges, r_good_edges)

ax.pcolormesh(
    T_good, R_good, X_good.T,
    cmap="RdBu_r",
    vmin=-1.5, vmax=1.5,
    shading="flat"
)

# ----------------------------
# Phase boundaries (piercing)
# ----------------------------
r_circle = r_bad_inner - 0.25

for ang in boundary_angles:
    ax.plot(
        [ang, ang],
        [r_circle, r_good_outer + 0.02],
        linestyle=":",
        color="gray",
        linewidth=2.4,
        alpha=0.8,
        zorder=10
    )

theta_dense = np.linspace(0, 2*np.pi, 512)

ax.plot(
    theta_dense,
    np.full_like(theta_dense, r_circle),
    linestyle=":",
    color="gray",
    linewidth=2.4,
    alpha=0.8,
    zorder=10
)

for r in [
    (r_bad_outer + r_good_inner) / 2,
    r_good_outer + 0.03,
    r_bad_inner - 0.03
]:
    ax.plot(
        theta_dense,
        np.full_like(theta_dense, r),
        color="gray",
        linewidth=3,
        alpha=1.0,
        zorder=10
    )

# ============================================================
# Optional: Phase labels (commented out for manual tweaking)
# ============================================================

# prepend start and append wrap-around end
# bounds = np.array(boundary_angles)
# bounds_ext = np.concatenate([bounds, [bounds[0] + 2*np.pi]])

# # mid-angle for each phase sector
# label_angles = []
# for i in range(len(bounds)):
#     a0 = bounds_ext[i]
#     a1 = bounds_ext[i + 1]
#     label_angles.append((a0 + a1) / 2.0)

# # radial position for text
# r_annot = r_bad_inner - 0.12

# for p, ang in zip(ordered_phases, label_angles):
#     ax.text(
#         ang,
#         r_annot,
#         p,
#         ha="center",
#         va="center",
#         fontsize=12,
#         fontweight="bold",
#         rotation=np.degrees(-ang + np.pi / 2),
#         rotation_mode="anchor",
#         color="black",
#         alpha=0.9,
#         zorder=20
#     )


# ----------------------------
# Polar aesthetics
# ----------------------------
ax.set_theta_direction(-1)
ax.set_theta_zero_location("N")
ax.set_ylim(0, 1.05)
ax.set_xticks([])
ax.set_yticks([])
ax.spines["polar"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
n_components = 6

svd = TruncatedSVD(n_components=n_components, random_state=0)
X_PCA = svd.fit_transform(emb.X)
V_PCA = emb.V @ svd.components_.T

print("[TPS] Fitting geometry spline …")
tps_pca = ThinPlateSpline(emb.X_emb)
tps_pca.fit(X_PCA, dof=30)

print("[TPS] Mapping vector field …")
vec_field = tps_pca.map_velocities(V_PCA)

print("[TPS] Fitting vector-field spline …")
tps_vf_pca = ThinPlateSpline(emb.X_emb)
tps_vf_pca.fit(vec_field, dof=30)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

# ------------------------------------------------------------
# 1) Use first 3 PCs
# ------------------------------------------------------------
selected_PC = np.array([0,1,2])

raw_expr  = np.asarray(X_PCA[:, selected_PC])
pred_expr = np.asarray(tps_pca.predict(emb.X_emb)[:, selected_PC])

# ------------------------------------------------------------
# 2) Cell cycle colors (continuous)
# ------------------------------------------------------------
phase = np.asarray(phase_numeric)

norm = mcolors.Normalize(vmin=phase.min(), vmax=phase.max())
cmap = plt.get_cmap("viridis")
cell_colors = cmap(norm(phase))


# ------------------------------------------------------------
# 3) Helper: polished 3D scatter
# ------------------------------------------------------------
# ------------------------------------------------------------
# Precompute PC scales (ONCE)
# ------------------------------------------------------------
singular_vals = svd.singular_values_
pc_scales = singular_vals[selected_PC]


def scatter3d(data, colors, title="",
              xlim_ref=None, ylim_ref=None, zlim_ref=None):

    fig = plt.figure(figsize=(6, 5), facecolor="white")
    ax = fig.add_subplot(111, projection="3d")

    x, y, z = data.T

    ax.scatter(
        x, y, z,
        c=colors,
        s=12,
        alpha=0.9,
        edgecolors="none"
    )

    ax.set_title(title, fontsize=12, pad=10)
    ax.view_init(elev=25, azim=-120)

    # ---- PCA-metric-correct geometry ----
    ax.set_box_aspect(pc_scales)

    # Optional: consistent limits
    if xlim_ref is not None:
        ax.set_xlim(xlim_ref)
    if ylim_ref is not None:
        ax.set_ylim(ylim_ref)
    if zlim_ref is not None:
        ax.set_zlim(zlim_ref)

    # ax.set_xlabel("PC1")
    # ax.set_ylabel("PC2")
    # ax.set_zlabel("PC3")

    ax.grid(False)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])

    plt.tight_layout()
    plt.show()

# ------------------------------------------------------------
# 4) Plot
# ------------------------------------------------------------
scatter3d(raw_expr,  cell_colors, "")
scatter3d(pred_expr, cell_colors, "")

In [ ]:
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "browser"  # <- THE KEY LINE

selected_PC = np.array([0, 1, 2])

raw_expr  = np.asarray(X_PCA[:, selected_PC])
pred_expr = np.asarray(tps_pca.predict(emb.X_emb)[:, selected_PC])
phase = np.asarray(phase_numeric)

def scatter3d_plotly(data, phase, title):
    fig = go.Figure(
        data=go.Scatter3d(
            x=data[:, 0],
            y=data[:, 1],
            z=data[:, 2],
            mode="markers",
            marker=dict(
                size=3,
                color=phase,
                colorscale="Viridis",
                opacity=0.9,
                colorbar=dict(title="Cell cycle phase"),
            )
        )
    )

    fig.update_layout(
        title=title,
        scene=dict(
            xaxis=dict(showticklabels=False),
            yaxis=dict(showticklabels=False),
            zaxis=dict(showticklabels=False),
            aspectmode="data",
        ),
        template="plotly_white",
        margin=dict(l=0, r=0, b=0, t=40),
    )

    fig.show()

scatter3d_plotly(raw_expr,  phase, "Raw PCA")
scatter3d_plotly(pred_expr, phase, "Reconstructed PCA")


In [ ]:
# angular coordinate for each cell (0, 2π)
n_cells = len(order)
theta = np.linspace(0, 2 * np.pi, n_cells, endpoint=False)

# phase labels in cell order
phases = np.array(adata.obs["cell_cycle_phase"])[order]
phase_order = ["S", "G2-M", "M", "M-G1", "G1-S"]

def circular_mean(theta):
    return np.arctan2(
        np.mean(np.sin(theta)),
        np.mean(np.cos(theta))
    ) % (2 * np.pi)

phase_angle = {}
for p in phase_order:
    mask = phases == p
    phase_angle[p] = circular_mean(theta[mask])

phase_order_sorted = sorted(
    phase_order,
    key=lambda p: phase_angle[p]
)

# unwrap angles relative to first phase
anchor = phase_order_sorted[0]
anchor_angle = phase_angle[anchor]

def unwrap(a):
    return (a - anchor_angle) % (2 * np.pi)

phase_angle_unwrapped = {
    p: unwrap(phase_angle[p]) for p in phase_order_sorted
}

boundaries = []

current_phases = []
for p in phase_order_sorted:
    current_phases.append(p)
    right_boundary = max(phase_angle_unwrapped[q] for q in current_phases)
    boundaries.append(right_boundary)

boundary_angles = [
    (b + anchor_angle) % (2 * np.pi) for b in boundaries
]

for ang in boundary_angles:
    ax.plot(
        [ang, ang],
        [r_bad_inner, r_good_outer],
        color="black",
        lw=1.5,
        alpha=0.6
    )

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# import seaborn as sns
# from scipy.ndimage import gaussian_filter1d
# from scipy.cluster.hierarchy import linkage, leaves_list

# X = emb.X_emb
# center = np.array([9, 13.5])

# # --- step 1: polar coordinates ---
# X_shifted = X - center
# r = np.sqrt((X_shifted**2).sum(axis=1))
# theta = np.arctan2(X_shifted[:,1], X_shifted[:,0])

# # ensure theta is in [0, 2pi)
# theta = (theta + 2*np.pi) % (2*np.pi)

# # --- step 2: compute max radius per angle bin ---
# n_bins = 100
# bins = np.linspace(0, 2*np.pi, n_bins+1)
# digitized = np.digitize(theta, bins) - 1

# max_r_per_bin = np.zeros(n_bins)
# for i in range(n_bins):
#     if np.any(digitized == i):
#         max_r_per_bin[i] = r[digitized == i].max()

# # expected max radius for each cell
# r_expected = max_r_per_bin[digitized]
# is_outer = r >= 0.5 * r_expected
# is_inner = ~is_outer

# # --- step 3: scatterplot colored by group ---
# plt.figure(figsize=(6,6))
# plt.scatter(X[is_inner,0], X[is_inner,1], s=10, c="steelblue", alpha=0.5, label="Inner half")
# plt.scatter(X[is_outer,0], X[is_outer,1], s=10, c="tomato", alpha=0.5, label="Outer half")
# plt.scatter(center[0], center[1], c="black", s=100, marker="x")
# plt.legend()
# plt.axis("equal")
# plt.title("Cells split by radius (inner vs outer)")
# plt.show()

In [ ]:
# shift embedding
X_shifted = emb.X_emb - center
r_data = np.linalg.norm(X_shifted, axis=1)
theta_data = np.arctan2(X_shifted[:,1], X_shifted[:,0])
theta_data = (theta_data + 2*np.pi) % (2*np.pi)

# angle bins
n_bins = 200
theta_bins = np.linspace(0, 2*np.pi, n_bins+1)
bin_idx = np.digitize(theta_data, theta_bins) - 1

# max radius per bin
r_max = np.zeros(n_bins)
for i in range(n_bins):
    mask = bin_idx == i
    r_max[i] = r_data[mask].max() if np.any(mask) else np.nan
    
from scipy.ndimage import gaussian_filter1d
r_max_smooth = gaussian_filter1d(np.nan_to_num(r_max, nan=np.nanmean(r_data)), sigma=5)

In [ ]:
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 50,
    "legend.fontsize": 14
})

plt.figure(figsize=(7,7))

# scatter embedding colored by relative position
plt.scatter(
    emb.X_emb[:,0], emb.X_emb[:,1],
    c=adata.obs["Cell_cycle_relativePos"].astype(float),
    cmap="viridis",
    s=400, alpha=0.3, edgecolors="none"
)

# spiral path
plt.plot(x_warped, y_warped, color="red", lw=10, label="Warped spiral")

# spiral center
# plt.scatter(center[0], center[1], c="black", marker="x", s=500, label="Center")

# formatting
plt.axis("equal")
plt.axis("off")
plt.title("Expression along Spiral Progression", pad=26)

# legend
# plt.legend(
#     bbox_to_anchor=(0.5, -0.08), loc="upper center",
#     ncol=2, frameon=False
# )

plt.tight_layout()
plt.show()


In [ ]:
plt.rcParams.update({
    "font.size": 32,        # base font size
    "axes.labelsize": 32,   # axis labels
    "axes.titlesize": 26,   # titles
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 32
})

plt.figure(figsize=(12,6))
ax = sns.heatmap(
    expr_clustered.T,
    cmap="RdBu_r", center=0,
    xticklabels=False, yticklabels=False,
    cbar_kws={"label": "z-scored expression"}
)

# --- format axes ---
ax.set_title("", fontsize=0)              # remove title
ax.set_ylabel("")                         # remove y-axis label
ax.set_xlabel("Spiral progression", fontsize=24, labelpad=16)

# make colorbar text larger
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=18)
cbar.set_label("z-scored expression", fontsize=22)

# --- add vertical lines for complete cycles ---
cycle_indices = []
theta0 = theta_spiral[0] % (2*np.pi)
for k in range(1, n_turns+1):
    cycle_angle = theta0 + k*2*np.pi
    idx = np.argmin(np.abs(theta_spiral - cycle_angle))
    cycle_indices.append(idx)

for idx in cycle_indices:
    ax.axvline(idx, color="black", lw=2.0, ls="--")

plt.tight_layout()
plt.show()


In [ ]:
# --- define well-known cell cycle genes ---
genes_to_plot = ["TOP2A", "CENPE", "NUF2", "CCNB1"]

# get indices (only if gene is in your filtered set)
gene_names_all = np.array(genes_filtered)[good_mask]
gene_indices = [np.where(gene_names_all == g)[0][0] for g in genes_to_plot if g in gene_names_all]
gene_names   = [gene_names_all[i] for i in gene_indices]

# --- shift phase so that it starts at 0 ---
phase_raw = theta_spiral / (2*np.pi)
phase = phase_raw - phase_raw[0]
phase = phase / phase.max() * n_turns

# setup grid: shrink vertical height per gene
fig, axes = plt.subplots(
    len(gene_indices), 1,
    figsize=(10, 1.6*len(gene_indices)),
    sharex=True
)

if len(gene_indices) == 1:
    axes = [axes]

for ax, idx, name in zip(axes, gene_indices, gene_names):
    y_raw = expr_along_spiral[:, idx]

    # shade 2 loops
    ax.axvspan(0, 1, color="orange", alpha=0.1)
    ax.axvspan(1, 2, color="green", alpha=0.1)

    # scatter expression
    ax.scatter(phase, y_raw, s=15, color="steelblue", alpha=0.7)

    # make gene name big & bold
    ax.set_title(name, fontsize=24, fontweight="bold", pad=6)
    ax.invert_xaxis()

    # remove y-axis ticks/labels
    ax.set_ylabel("")
    ax.set_yticks([])

# shared axis labels
fig.text(0.04, 0.5, "Gene expression", va="center", rotation="vertical", fontsize=18)
axes[-1].set_xlabel("Spiral phase (cycles)", fontsize=18)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d

# --- extract data ---
x = np.array(r2["expr_corr_gene"])   # Pearson r (expression)
y = np.array(r2["vel_corr_gene"])    # Cosine similarity (velocity)

mask = np.isfinite(x) & np.isfinite(y)
x, y = x[mask], y[mask]
gene_names = np.array(adata.var_names[global_mask])[mask]

# --- pick top 4 genes by combined score (expr * vel correlation) ---
combined_score = x * y
top_idx = np.argsort(combined_score)[-4:][::-1]  # top 4, descending
top_genes = gene_names[top_idx]

print("Top 4 genes:", top_genes)